# Advanced RAG

## Basic Prerequisite

### Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*is in beta.*")
warnings.filterwarnings("ignore", message=".*allowed_objects.*")

In [ ]:
import os
import hashlib
from typing import TypedDict, Literal, List

import numpy as np
from sklearn.cluster import KMeans

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.documents import Document
from langchain_core.load import dumps, loads
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

### Define all paths

In [ ]:
DB_DIR = "./chroma_db"
SOURCE_PATH = "./Pdfs/AI_Engineering.pdf"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
MAX_RETRIES = 2

MODEL = ChatGroq(
    model_name="openai/gpt-oss-120b",
    temperature=0.0,
)

Embedding = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'}
)
USE_ADVANCED_INDEXING = False

### Hash coding & Indexing

In [ ]:
# Get hash code of source doc, used to skip re-embedding unchanged files
def get_file_hash(file_path):
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()


assert os.path.exists(SOURCE_PATH), f"Please place your PDF file at: {SOURCE_PATH}"

In [ ]:
db_exists = os.path.exists(DB_DIR) and os.listdir(DB_DIR)

if db_exists:
    print("Found existing vector store. Loading from disk...")

    vectorstore = Chroma(
        persist_directory=DB_DIR,
        embedding_function=Embedding
    )
    all_docs = vectorstore.get()

    existing_hashes = set()
    if all_docs and 'metadatas' in all_docs:
        existing_hashes = {meta.get('file_hash') for meta in all_docs['metadatas'] if meta}

    current_hash = get_file_hash(SOURCE_PATH)

    if current_hash in existing_hashes:
        print("Smart Check: This exact file content has already been vectorized. Loading existing database...")
    else:
        print("Smart Check: New file detected! Appending new content to existing database...")
        loader = PyPDFLoader(SOURCE_PATH)
        docs = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP)
        splits = text_splitter.split_documents(docs)

        for split in splits:
            split.metadata['file_hash'] = current_hash

        vectorstore.add_documents(splits)
        print("✅ Added new file data successfully!")

else:
    print("Database not found. Building initial vector store...")

    loader = PyPDFLoader(SOURCE_PATH)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP)
    splits = text_splitter.split_documents(docs)

    current_hash = get_file_hash(SOURCE_PATH)
    for split in splits:
        split.metadata['file_hash'] = current_hash

    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=Embedding,
        persist_directory=DB_DIR
    )
    print("✅ Successfully built, saved, and loaded the new vector store!")

<h3> Retriever & content resolution

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

def resolve_content(doc: Document) -> str:
    """Return the real content for a retrieved doc -- the source chunk if this
    is a multi-query alias, otherwise the doc's own content (base chunk or RAPTOR summary).
    Every place that turns a retrieved Document into context text must go through this,
    or a multi-query paraphrase can leak into the LLM's context as if it were real content."""
    return doc.metadata.get("source_chunk", doc.page_content)

<h3> Raptor Indexing

In [ ]:
class RaptorIndexer:
    """Builds a hierarchical summary tree over a set of text chunks and writes
    each summary node into the vector store."""

    def __init__(self, model, embedding_model, vectorstore, max_levels=3, min_cluster_size=4):
        self.model = model
        self.embedding_model = embedding_model
        self.vectorstore = vectorstore
        self.max_levels = max_levels
        self.min_cluster_size = min_cluster_size
        self._build_summary_chain()

    def _build_summary_chain(self):
        summary_template = """Summarize the following passages into a single coherent paragraph.
Preserve the specific facts and terms -- do not generalize them away.

Passages:
{text}

Summary:"""
        self._summarize = (
            ChatPromptTemplate.from_template(summary_template) | self.model | StrOutputParser()
        )

    def _n_clusters(self, n_texts: int) -> int:
        return min(max(2, n_texts // self.min_cluster_size), n_texts - 1, 5)

    def _cluster(self, texts: list[str]) -> np.ndarray:
        embeddings = np.array(self.embedding_model.embed_documents(texts))
        n_clusters = self._n_clusters(len(texts))
        return KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit_predict(embeddings)

    def _summarize_cluster(self, texts: list[str]) -> str:
        return self._summarize.invoke({"text": "\n\n".join(texts)})

    def _add_summary(self, summary: str, level: int, file_hash: str):
        self.vectorstore.add_texts(
            texts=[summary],
            metadatas=[{"node_type": "raptor_summary", "level": level, "file_hash": file_hash}],
        )

    def build(self, texts: list[str], file_hash: str, level: int = 1):
        if len(texts) <= self.min_cluster_size or level > self.max_levels:
            summary = self._summarize_cluster(texts)
            self._add_summary(summary, level, file_hash)
            print(f"→ RAPTOR level {level}: 1 root summary (stopping)")
            return

        labels = self._cluster(texts)
        summaries = []
        for cluster_id in sorted(set(labels)):
            cluster_texts = [t for t, l in zip(texts, labels) if l == cluster_id]
            summary = self._summarize_cluster(cluster_texts)
            self._add_summary(summary, level, file_hash)
            summaries.append(summary)

        print(f"→ RAPTOR level {level}: {len(summaries)} cluster summaries")

        if len(summaries) > 1:
            self.build(summaries, file_hash, level=level + 1)

<h3> MultiQueryIndexer

In [ ]:
class MultiQueryIndexer:
    """Adds paraphrased aliases of each chunk to the vector store, pointing back
    to the original chunk via metadata."""

    def __init__(self, model, vectorstore, phrasings_per_chunk=2):
        self.model = model
        self.vectorstore = vectorstore
        self.phrasings_per_chunk = phrasings_per_chunk
        self._build_chain()

    def _build_chain(self):
        phrasing_template = """Write {n} short, differently-worded questions that the
following passage would be a good answer to. One per line, no numbering.

Passage:
{text}"""
        self._generate_phrasings = (
            ChatPromptTemplate.from_template(phrasing_template)
            | self.model | StrOutputParser()
            | (lambda x: [q for q in x.split("\n") if q.strip()])
        )

    def index(self, chunks: list[str], file_hash: str):
        for chunk in chunks:
            phrasings = self._generate_phrasings.invoke({"text": chunk, "n": self.phrasings_per_chunk})
            self.vectorstore.add_texts(
                texts=phrasings,
                metadatas=[
                    {"node_type": "multi_query_alias", "source_chunk": chunk, "file_hash": file_hash}
                    for _ in phrasings
                ],
            )
        print(f"→ Multi-query: added aliases for {len(chunks)} chunks")

In [ ]:
all_data = vectorstore.get()
base_chunks = [
    text for text, meta in zip(all_data["documents"], all_data["metadatas"])
    if not meta.get("node_type")
]

raptor_exists = any(m.get("node_type") == "raptor_summary" for m in all_data["metadatas"])
multi_query_exists = any(m.get("node_type") == "multi_query_alias" for m in all_data["metadatas"])

raptor = RaptorIndexer(model=MODEL, embedding_model=Embedding, vectorstore=vectorstore)
multi_query = MultiQueryIndexer(model=MODEL, vectorstore=vectorstore)

if raptor_exists:
    print("RAPTOR summaries already exist for this file, skipping")
else:
    if USE_ADVANCED_INDEXING:
        raptor.build(base_chunks, file_hash=current_hash)

if multi_query_exists:
    print("Multi-query aliases already exist for this file, skipping")
else:
    if USE_ADVANCED_INDEXING:
        multi_query.index(base_chunks, file_hash=current_hash)

## Query Router

Structured output instead of free-text parsing -- the model returns a typed `route`, no regex needed to figure out what it meant.

In [ ]:
class RouteQuery(BaseModel):
    """Route a user question to the correct branch."""

    route: Literal["needs_clarification", "index_lookup", "web_search"] = Field(
        description=(
            "needs_clarification: question is ambiguous or missing context. "
            "index_lookup: question can be answered from the indexed documents. "
            "web_search: question is about current events or anything outside the indexed documents."
        )
    )

structured_router = MODEL.with_structured_output(RouteQuery)

router_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert at routing a user question. "
                "Classify it as needs_clarification, index_lookup, or web_search."),
    ("human", "{question}"),
])

route_chain = router_prompt | structured_router

<h3>Clarification branch</h3>

Direct LLM response asking a follow-up -- no retrieval involved, kept simple on purpose.

In [ ]:
clarify_prompt = ChatPromptTemplate.from_messages([
    ("system", "The user's question is ambiguous or missing context. "
                "Ask a short, specific follow-up question to clarify what they mean."),
    ("human", "{question}"),
])

clarify_chain = clarify_prompt | MODEL | StrOutputParser()

## QueryTransformation class

In [ ]:
class QueryTransformation:
    """Query rewriting strategies for retrieval. One instance holds the model,
    retriever, and all prebuilt prompt chains so individual calls stay cheap."""

    def __init__(self, model, retriever):
        self.model = model
        self.retriever = retriever
        self._build_chains()

    def _build_chains(self):
        rag_fusion_template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
        self._rag_fusion_queries = (
            ChatPromptTemplate.from_template(rag_fusion_template)
            | self.model | StrOutputParser()
            | (lambda x: [q for q in x.split("\n") if q.strip()])
        )

        decomposition_template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answered in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
        self._sub_questions = (
            ChatPromptTemplate.from_template(decomposition_template)
            | self.model | StrOutputParser()
            | (lambda x: [q for q in x.split("\n") if q.strip()])
        )

        step_back_examples = [
            {"input": "Could the members of The Police perform lawful arrests?",
             "output": "what can the members of The Police do?"},
            {"input": "Jan Sindel's was born in what country?",
             "output": "what is Jan Sindel's personal history?"},
        ]
        few_shot_prompt = FewShotChatMessagePromptTemplate(
            example_prompt=ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")]),
            examples=step_back_examples,
        )
        step_back_prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert at world knowledge. Your task is to step back and "
                        "paraphrase a question to a more generic step-back question, which is "
                        "easier to answer. Here are a few examples:"),
            few_shot_prompt,
            ("user", "{question}"),
        ])
        self._step_back_query = step_back_prompt | self.model | StrOutputParser()

        hyde_template = """Please write a passage to answer the question
Question: {question}
Passage:"""
        self._hyde_doc = ChatPromptTemplate.from_template(hyde_template) | self.model | StrOutputParser()

    @staticmethod
    def _unique_union(document_lists: list[list]):
        flattened = [dumps(doc) for sublist in document_lists for doc in sublist]
        return [loads(doc, allowed_objects=[Document]) for doc in set(flattened)]

    def rag_fusion(self, question: str) -> list:
        """Best for e-commerce or customer-support style questions comparing/finding options."""
        queries = self._rag_fusion_queries.invoke({"question": question})
        results = [self.retriever.invoke(q) for q in queries]
        fused_scores = {}
        for docs in results:
            for rank, doc in enumerate(docs):
                doc_str = dumps(doc)
                fused_scores[doc_str] = fused_scores.get(doc_str, 0) + 1 / (rank + 60)
        reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        return [loads(doc, allowed_objects=[Document]) for doc, score in reranked]

    def decomposition(self, question: str) -> list:
        """Best for research-based, complex multi-part questions."""
        sub_questions = self._sub_questions.invoke({"question": question})
        doc_lists = [self.retriever.invoke(sq) for sq in sub_questions]
        return self._unique_union(doc_lists)

    def step_back(self, question: str) -> list:
        """Best for code debugging / troubleshooting questions needing broader context first."""
        step_back_question = self._step_back_query.invoke({"question": question})
        normal_docs = self.retriever.invoke(question)
        step_back_docs = self.retriever.invoke(step_back_question)
        return self._unique_union([normal_docs, step_back_docs])

    def hyde(self, question: str) -> list:
        """Best for FAQ / help-doc style questions with a fairly direct expected answer."""
        hypothetical_doc = self._hyde_doc.invoke({"question": question})
        return self.retriever.invoke(hypothetical_doc)

    def direct(self, question: str) -> list:
        """Best for simple, unambiguous questions that don't need any rewriting."""
        return self.retriever.invoke(question)

    @property
    def strategies(self) -> dict:
        return {
            "rag_fusion": self.rag_fusion, "decomposition": self.decomposition,
            "step_back": self.step_back, "hyde": self.hyde, "direct": self.direct,
        }

    def retrieve(self, question: str, strategy: str) -> list:
        """Dispatch to the named strategy."""
        return self.strategies[strategy](question)


qt = QueryTransformation(model=MODEL, retriever=retriever)

<h3> Strategy Selector

Classifies the question once, picks a single strategy. On retry, previously-tried strategies are excluded so it doesn't repeat a failure.

In [ ]:
class SelectStrategy(BaseModel):
    """Pick the single best query-transformation strategy for this question."""

    strategy: Literal["rag_fusion", "decomposition", "step_back", "hyde", "direct"] = Field(
        description=(
            "rag_fusion: e-commerce or customer-support style questions comparing/finding options. "
            "decomposition: research-based, complex multi-part questions. "
            "step_back: code debugging or troubleshooting questions needing broader context first. "
            "hyde: FAQ or help-doc style questions with a fairly direct expected answer. "
            "direct: simple, unambiguous questions that don't need any query rewriting."
        )
    )

structured_selector = MODEL.with_structured_output(SelectStrategy)

select_strategy_prompt = ChatPromptTemplate.from_messages([
    ("system", "Choose the single best query-transformation strategy for this question. "
                "Do not choose any of these already-tried strategies: {tried}"),
    ("human", "{question}"),
])

select_chain = select_strategy_prompt | structured_selector

<h3> Retrieval Grader

In [ ]:
class GradeRetrieval(BaseModel):
    """Grade whether retrieved documents are relevant to the question."""

    is_relevant: bool = Field(
        description="True if the documents contain information that helps answer the question."
    )

structured_grader = MODEL.with_structured_output(GradeRetrieval)

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a grader checking whether retrieved documents are relevant to a question."),
    ("human", "Question: {question}\n\nRetrieved documents:\n{documents}"),
])

grade_chain = grade_prompt | structured_grader

## WebRetriever class

Same shape as `QueryTransformation`: rewrites the question into a search-engine-friendly query, then runs Tavily.

In [ ]:
class WebRetriever:
    """Rewrites a question into a search query and runs it through Tavily."""

    def __init__(self, model, search_tool):
        self.model = model
        self.search_tool = search_tool
        self._build_chain()

    def _build_chain(self):
        query_template = """Rewrite this question as a concise web search query (keywords, not a full sentence).
Question: {question}
Search query:"""
        self._generate_query = (
            ChatPromptTemplate.from_template(query_template) | self.model | StrOutputParser()
        )

    def generate_query(self, question: str) -> str:
        """Rewrites the user question into a search-engine-friendly query."""
        return self._generate_query.invoke({"question": question}).strip()

    def search(self, query: str) -> list:
        """Runs the web search and returns a list of content strings."""
        results = self.search_tool.invoke(query)
        return [r["content"] for r in results if "content" in r]


web = WebRetriever(model=MODEL, search_tool=TavilySearchResults(max_results=3))

## Self-Correction loops

<h3> Generation chain

In [ ]:
gen_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the provided context. "
                "If the context doesn't contain the answer, say so clearly -- do not guess."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])
answer_chain = gen_prompt | MODEL | StrOutputParser()

# Used on hallucination retry -- same context, stricter instruction
strict_gen_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY facts explicitly stated in the context. "
                "Do not add, infer, or generalize beyond what is written. "
                "If the context doesn't contain the answer, say so clearly."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])
strict_answer_chain = strict_gen_prompt | MODEL | StrOutputParser()

<h3> Hallucination grader

In [ ]:
class GradeHallucination(BaseModel):
    """Grade whether an answer is grounded in the provided documents."""

    is_grounded: bool = Field(
        description="True if every claim in the answer is supported by the documents. "
                     "False if the answer adds, infers, or generalizes beyond the documents."
    )

structured_hallucination_grader = MODEL.with_structured_output(GradeHallucination)

hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict grader checking whether an answer is fully grounded "
                "in a set of documents. Flag anything the answer states that the documents "
                "do not explicitly support."),
    ("human", "Documents:\n{documents}\n\nAnswer:\n{generation}"),
])

hallucination_grade_chain = hallucination_prompt | structured_hallucination_grader

<h3> Sufficiency grader

In [ ]:
class GradeSufficiency(BaseModel):
    """Grade whether an answer actually addresses the question."""

    is_sufficient: bool = Field(
        description="True if the answer meaningfully addresses the question. "
                     "False if it's a non-answer, an unnecessary refusal, or only partially answers."
    )

structured_sufficiency_grader = MODEL.with_structured_output(GradeSufficiency)

sufficiency_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a grader checking whether an answer actually addresses the question asked."),
    ("human", "Question: {question}\n\nAnswer:\n{generation}"),
])

sufficiency_grade_chain = sufficiency_prompt | structured_sufficiency_grader

## LangGraph

<h3> State Schema

In [ ]:
class RAGState(TypedDict):
    question: str
    generation: str
    route: str
    strategy: str
    tried_strategies: List[str]
    web_query: str
    documents: List[str]
    is_relevant: bool
    rewrite_count: int
    is_grounded: bool
    hallucination_count: int
    is_sufficient: bool
    insufficiency_count: int


def initial_state(question: str) -> RAGState:
    """Fresh state for a new question -- avoids repeating this 13-field dict at every call site."""
    return {
        "question": question, "generation": "", "route": "", "strategy": "",
        "tried_strategies": [], "web_query": "", "documents": [], "is_relevant": False,
        "rewrite_count": 0, "is_grounded": False, "hallucination_count": 0,
        "is_sufficient": False, "insufficiency_count": 0,
    }

<h3> Nodes

Every node here now only references chains/classes already defined above it -- this ordering is what makes the notebook runnable top to bottom.

In [ ]:
def analyze_query(state: RAGState) -> RAGState:
    result = route_chain.invoke({"question": state["question"]})
    print(f"→ Routed to: {result.route}")
    return {**state, "route": result.route}


def clarify(state: RAGState) -> RAGState:
    answer = clarify_chain.invoke({"question": state["question"]})
    return {**state, "generation": answer}


def select_strategy(state: RAGState) -> RAGState:
    tried = state.get("tried_strategies", [])
    result = select_chain.invoke({
        "question": state["question"],
        "tried": ", ".join(tried) if tried else "none",
    })
    print(f"→ Strategy chosen: {result.strategy}")
    return {**state, "strategy": result.strategy}


def generate_strategy(state: RAGState) -> RAGState:
    docs = qt.retrieve(state["question"], state["strategy"])
    return {**state, "documents": [resolve_content(d) for d in docs]}


def generate_web_query(state: RAGState) -> RAGState:
    query = web.generate_query(state["question"])
    print(f"→ Web query: {query}")
    return {**state, "web_query": query}


def run_web_search(state: RAGState) -> RAGState:
    documents = web.search(state["web_query"])
    return {**state, "documents": documents}


def grade_retrieval(state: RAGState) -> RAGState:
    result = grade_chain.invoke({
        "question": state["question"],
        "documents": "\n\n".join(state["documents"]),
    })
    print(f"→ Relevant: {result.is_relevant}")

    if result.is_relevant:
        return {**state, "is_relevant": True}

    tried = state["tried_strategies"] + [state["strategy"]] if state["route"] == "index_lookup" else state["tried_strategies"]
    return {
        **state,
        "is_relevant": False,
        "rewrite_count": state["rewrite_count"] + 1,
        "tried_strategies": tried,
    }


def generate_answer(state: RAGState) -> RAGState:
    chain = strict_answer_chain if state["hallucination_count"] > 0 else answer_chain
    answer = chain.invoke({
        "context": "\n\n".join(state["documents"]),
        "question": state["question"],
    })
    return {**state, "generation": answer}


def grade_hallucination(state: RAGState) -> RAGState:
    result = hallucination_grade_chain.invoke({
        "documents": "\n\n".join(state["documents"]),
        "generation": state["generation"],
    })
    print(f"→ Grounded: {result.is_grounded}")

    if result.is_grounded:
        return {**state, "is_grounded": True}
    return {**state, "is_grounded": False, "hallucination_count": state["hallucination_count"] + 1}

def grade_sufficiency(state: RAGState) -> RAGState:
    result = sufficiency_grade_chain.invoke({
        "question": state["question"],
        "generation": state["generation"],
    })
    print(f"→ Sufficient: {result.is_sufficient}")

    if result.is_sufficient:
        return {**state, "is_sufficient": True}
    return {**state, "is_sufficient": False, "insufficiency_count": state["insufficiency_count"] + 1}


def re_retrieve(state: RAGState) -> RAGState:
    # Insufficiency often means the question has parts the first retrieval missed --
    # decomposition splits it into sub-questions and retrieves for each separately.
    docs = qt.decomposition(state["question"])
    return {**state, "documents": [resolve_content(d) for d in docs]}


def decide_route(state: RAGState) -> str:
    return state["route"]


def decide_after_grade(state: RAGState) -> str:
    if state["is_relevant"] or state["rewrite_count"] >= MAX_RETRIES:
        return "proceed"
    return "retry_web" if state["route"] == "web_search" else "retry_index"


def decide_after_hallucination(state: RAGState) -> str:
    if state["is_grounded"] or state["hallucination_count"] >= MAX_RETRIES:
        return "check_sufficiency"
    return "regenerate"


def decide_after_sufficiency(state: RAGState) -> str:
    if state["is_sufficient"] or state["insufficiency_count"] >= MAX_RETRIES:
        return "done"
    return "re_retrieve"

<h3> Build the graph

In [ ]:
graph = StateGraph(RAGState)

graph.add_node("analyze_query", analyze_query)
graph.add_node("clarify", clarify)
graph.add_node("select_strategy", select_strategy)
graph.add_node("generate_strategy", generate_strategy)
graph.add_node("generate_web_query", generate_web_query)
graph.add_node("run_web_search", run_web_search)
graph.add_node("grade_retrieval", grade_retrieval)
graph.add_node("generate_answer", generate_answer)
graph.add_node("grade_hallucination", grade_hallucination)
graph.add_node("grade_sufficiency", grade_sufficiency)
graph.add_node("re_retrieve", re_retrieve)

graph.add_edge(START, "analyze_query")
graph.add_conditional_edges(
    "analyze_query",
    decide_route,
    {
        "needs_clarification": "clarify",
        "index_lookup": "select_strategy",
        "web_search": "generate_web_query",
    },
)
graph.add_edge("clarify", END)

graph.add_edge("select_strategy", "generate_strategy")
graph.add_edge("generate_strategy", "grade_retrieval")

graph.add_edge("generate_web_query", "run_web_search")
graph.add_edge("run_web_search", "grade_retrieval")

graph.add_conditional_edges(
    "grade_retrieval",
    decide_after_grade,
    {
        "retry_index": "select_strategy",
        "retry_web": "generate_web_query",
        "proceed": "generate_answer",
    },
)

graph.add_edge("generate_answer", "grade_hallucination")
graph.add_conditional_edges(
    "grade_hallucination",
    decide_after_hallucination,
    {
        "regenerate": "generate_answer",
        "check_sufficiency": "grade_sufficiency",
    },
)

graph.add_conditional_edges(
    "grade_sufficiency",
    decide_after_sufficiency,
    {
        "re_retrieve": "re_retrieve",
        "done": END,
    },
)
graph.add_edge("re_retrieve", "generate_answer")

app = graph.compile()

### Visualize the graph (optional)

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Skipping graph visualization: {e}")

## Test end to end

Covers all 3 routes, plus a deliberately hard question meant to exercise the strategy-retry loop.

In [ ]:
test_questions = [
    "What is the difference between a foundation model and a large language model?",
    # index_lookup, expect direct/hyde -- simple factual definition

    "What are the main approaches to evaluating AI systems, and how does evaluating the base model differ from evaluating the full application?",
    # index_lookup, expect decomposition -- multi-part, research-style question

    "My RAG pipeline keeps retrieving irrelevant chunks even though the embeddings look fine, what's likely going wrong?",
    # index_lookup, expect step_back -- troubleshooting framing

    "What is prompt engineering?",
    # index_lookup, expect hyde/direct -- FAQ-style, single concept

    "How do finetuning and RAG compare as ways to adapt a foundation model to a specific task?",
    # index_lookup, expect rag_fusion -- comparing/choosing between options

    "What new foundation models or inference optimization techniques have been released this month?",
    # web_search -- explicitly time-bound, outside any static book's content

    "Is that approach actually better for my use case?",
    # needs_clarification -- "that" and "my use case" are both undefined
]

for q in test_questions:
    print(f"{'=' * 60}\nQ: {q}")
    result = app.invoke(initial_state(q))
    print(f"A: {result['generation']}\n")